# ML Assignment
BEN BELHASSEN Mohamed Ali (102291), CIAMPANA Lorenzo (102296), FILESI Gianluca (102299), NUBE Giacomo (102311)

In [1]:
import pandas as pd
path = '/Users/gianlucafilesi/Library/CloudStorage/OneDrive-EDHEC/01 MACHINE LEARNING/Machine Learning/training_dataset.xlsx'
DF = pd.read_excel(path)

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold


from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
# import xgboost as xgb


from sklearn.pipeline import make_pipeline

from sklearn.preprocessing import StandardScaler


from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier


# Data cleaning and processing

In [4]:
DF.describe()

,URBRRL,RATCAT_A,INCGRP_A,INCTCFLG_A,FAMINCTC_A,IMPINCFLG_A,RJWKCLSOFT_A,RJWCLSNOSD_A,RJWRKCLSSD_A,RECJOBSD_A,...,HYPEV_A,PHSTAT_A,PROXYREL_A,PROXY_A,AVAIL_A,HHSTAT_A,INTV_MON,RECTYPE,WTFA_A,POVRATTC_A
count,20340.000000,20340.000000,20340.000000,20340.000000,20340.000000,20340.000000,488.000000,346.000000,346.000000,838.000000,...,20340.000000,20340.000000,240.000000,244.000000,20340.000000,20340.0,20340.000000,20340.0,20340.000000,20340.000000
mean,2.317650,10.131858,3.119371,0.036185,81494.078417,0.363668,2.459016,2.497110,2.858382,1.688544,...,1.650934,2.339676,1.262500,1.016393,1.143363,1.0,6.820108,10.0,7865.011012,4.403138
std,1.054041,3.895839,1.579656,0.186754,63031.416044,0.710902,1.609252,1.420856,1.172215,0.993233,...,0.533809,1.047854,0.615227,0.127244,0.919118,0.0,3.534833,0.0,6536.120255,2.998080
min,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,10.0,396.210000,0.000000
25%,1.000000,8.000000,1.000000,0.000000,34000.000000,0.000000,1.000000,1.000000,2.000000,1.000000,...,1.000000,2.000000,1.000000,1.000000,1.000000,1.0,3.000000,10.0,3666.176500,2.030000
50%,2.000000,11.000000,3.000000,0.000000,65000.000000,0.000000,2.000000,3.000000,3.000000,2.000000,...,2.000000,2.000000,1.000000,1.000000,1.000000,1.0,8.000000,10.0,6226.949500,3.750000
75%,3.000000,14.000000,5.000000,0.000000,110000.000000,0.000000,4.000000,4.000000,4.000000,2.000000,...,2.000000,3.000000,1.000000,1.000000,1.000000,1.0,10.000000,10.0,9777.649500,6.060000
max,4.000000,14.000000,5.000000,1.000000,250000.000000,2.000000,9.000000,9.000000,4.000000,9.000000,...,9.000000,9.000000,4.000000,2.000000,8.000000,1.0,12.000000,10.0,91832.051000,11.000000


### Features selection - phase 1

In [5]:
DF['HHX'].value_counts()

HHX
H050728    1
H005968    1
H062365    1
H023412    1
H043314    1
          ..
H014906    1
H065900    1
H038525    1
H030009    1
H055546    1
Name: count, Length: 20340, dtype: int64

Since 'HHX' are labels, we can remove them.

In [6]:
df = DF.copy()
df.drop('HHX', axis=1, inplace=True)

When a column is filled by the same value, it is not influent. We remove it.

In [18]:
for each in df.columns:
    if df[each].nunique() == 1:
        df.drop(each, axis=1, inplace=True)

### NaN values - removal and replacement

We have some columns with NaN. When the number of empty rows is much higher than the unempty ones, that feature is not useful. For this reason, we prefer removing it. We have set as thresold the $10\%$ of the dataset length.

In [49]:
features = df.columns

for feature in features:
    if df[feature].isna().sum()> 0.1*len(df):
        df.drop(feature, axis=1, inplace=True)
features = df.columns

For the other features, we can fill the NaN using the K-neighbors classifier.

In [50]:
imputer = KNNImputer(n_neighbors=3)
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

In [51]:
df_imputed.head()

,URBRRL,RATCAT_A,INCGRP_A,INCTCFLG_A,FAMINCTC_A,IMPINCFLG_A,PPSU,PSTRAT,HISPALLP_A,RACEALLP_A,...,MIEV_A,ANGEV_A,CHDEV_A,CHLEV_A,HYPEV_A,PHSTAT_A,AVAIL_A,INTV_MON,WTFA_A,POVRATTC_A
0,2.0,11.0,3.0,0.0,70000.0,0.0,2.0,107.0,2.0,1.0,...,2.0,2.0,2.0,2.0,2.0,3.0,1.0,3.0,13772.434,3.50
1,4.0,6.0,1.0,0.0,30000.0,0.0,73.0,109.0,2.0,1.0,...,2.0,2.0,2.0,2.0,2.0,1.0,1.0,7.0,9326.686,1.70
2,3.0,11.0,3.0,0.0,63464.0,0.0,65.0,124.0,2.0,1.0,...,2.0,2.0,2.0,2.0,2.0,1.0,1.0,2.0,4509.724,3.60
3,3.0,5.0,1.0,0.0,25000.0,2.0,14.0,125.0,2.0,1.0,...,2.0,2.0,2.0,2.0,1.0,3.0,1.0,11.0,10849.226,1.46
4,4.0,13.0,4.0,0.0,75000.0,0.0,20.0,146.0,2.0,1.0,...,2.0,2.0,2.0,2.0,1.0,4.0,1.0,7.0,5771.964,4.85


We check that no column has issue.

In [52]:
Sigma = df_imputed[features].corr().abs()
Sigma.values.diagonal().sum() == len(features)

np.True_

Since the sum of the correlation diagonal is equal to the number of varibles, it means that there are not issues.

### Multicollinearity

We remove the columns whose correlation coefficients is too close to 1, to avoid multicollinerarity issue.

In [53]:
np.fill_diagonal(Sigma.values, 0)
for each in Sigma.columns:
    if Sigma[each].max() > 0.9:
        df_imputed.drop(each, axis=1, inplace=True)
features = df_imputed.columns

In [54]:
Sigma = df_imputed[features].corr().abs()

Now we have a look for multicollinearity issue.

In [55]:
target = ['WEIGHTLBTC_A']
features.drop(target)
X = df_imputed[features]
y = df_imputed[target].values.ravel()
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_scaled['Intercept'] = 1

In [56]:
def calculate_vif(X):
    vif_data = pd.DataFrame()
    vif_data["Variable"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X, i) for i in range(X.shape[1])]
    return vif_data

In [57]:
vif_data = calculate_vif(X_scaled)
vif_data[vif_data['VIF'] > 10]

,Variable,VIF
20,PA18_05R_A,11.832944
21,PA18_02R_A,16.063993
23,MODNR_A,12.823709
24,MODTPR_A,11.925346
25,VIGNR_A,17.169554
26,VIGTPR_A,22.906527
27,VIGFREQW_A,11.375012
28,STRTPR_A,27.272205
29,STRNR_A,22.540144
30,STRFREQW_A,14.356124


In [58]:
while vif_data['VIF'].max() > 10:
    max_vif_variable = vif_data.loc[vif_data['VIF'].idxmax(), 'Variable']
    X_scaled = X_scaled.drop(columns=[max_vif_variable])
    vif_data = calculate_vif(X_scaled)
    
vif_data.max()
X_scaled.drop('Intercept', axis=1, inplace=True)


### Dimensions reduction

We set as thresold the $80\%$ of the sum of the variances.

In [59]:
Sigma = X_scaled.corr()
L, V = np.linalg.eig(Sigma)
L = np.sort(L)[::-1]

In [60]:
i = 1
S = 0
while S < 0.8:
    S = L[:i].sum() / L.sum()
    i += 1

In [61]:
pca = PCA(n_components=i-1)
data_pca = pca.fit_transform(X_scaled)
data_pca.shape

(20340, 66)

# Models

In [ ]:
feature_train,feature_test,target_train,target_test = train_test_split(X_scaled,y, test_size=0.3)

In [ ]:
def model_MSE(model):
    model.fit(feature_train, target_train)
    target_pred = model.predict(feature_test)

    MSE = mean_squared_error(target_test, target_pred)
    return MSE

In [ ]:
model = DecisionTreeClassifier()
model_MSE(model)

0.0014749262536873156

In [ ]:
model = BaggingClassifier()
model_MSE(model)

0.0016388069485414618

In [ ]:
model = RandomForestClassifier()
model_MSE(model)

164.80498197312357

In [ ]:
model = AdaBoostClassifier()
model_MSE(model)

792.9906588003934

In [ ]:
model = GradientBoostingClassifier()
model_MSE(model)

27.15765322844969

## XGBRegressor : the smallest RMSE

In [ ]:
model = xgb.XGBRegressor()
model_MSE(model)


0.0012924101865255605

## LR : smallest RMSE but OVERFITTING

In [30]:
model = LinearRegression()
model_MSE(model)

1.1021543635085342e-26

In [31]:
model = Lasso(alpha=1)
model_MSE(model)

1.0038469662872695

In [32]:
model = Ridge(alpha=1)
model_MSE(model)

2.7280811458556538e-05

## PCA : unecessary

In [62]:
feature_train,feature_test,target_train,target_test = train_test_split(data_pca,y, test_size=0.3)

In [63]:
model = DecisionTreeClassifier()
model_MSE(model)

909.2159947558177

In [64]:
model = BaggingClassifier()
model_MSE(model)

888.7763028515241

In [65]:
model = xgb.XGBRegressor()
model_MSE(model)

217.00288181920618

In [66]:
model = LinearRegression()
model_MSE(model)

151.70809513348573

In [67]:
model = Lasso(alpha=1)
model_MSE(model)

192.08597092977467

In [68]:
model = Ridge(alpha=1)
model_MSE(model)

151.70828413513925

In [34]:
model = DecisionTreeClassifier()
model.fit(feature_train, target_train)
target_pred = model.predict(feature_test)

MSE = mean_squared_error(target_test, target_pred)
MSE

937.6643723369388